# 02 · Confirm one temperature target change

Choose a modest approved `TARGET_C`. The operation displays identity and state, then requires the exact displayed `SET ...` token before sending one setting. Any other response cancels. Matching rounded readback establishes the target, not thermal settling. No previous target is automatically restored.

Read [the tutorial guide](README.md) and [physical commissioning](../docs/commissioning.md) first. Use one serial owner. These notebook outputs are intentionally empty; no bench result is implied.


## Environment

From the repository root, run:

```powershell
uv sync --locked
uv run --locked --with jupyterlab --with ipykernel python -m jupyterlab
```

Choose that environment's Python kernel and run top to bottom. Restart the kernel after driver changes. The next cell opens no serial port.


In [ ]:
import sys
from pathlib import Path

# Prefer this checkout when launched from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, REPO_ROOT.parent):
    if (candidate / "src/tark_nrc/controller.py").is_file():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
print(f"Python: {sys.executable}")
print(f"Repository/config base: {REPO_ROOT}")

## Imports


In [ ]:
from math import isfinite
from pathlib import Path

from tark_nrc import ProtocolError, create_chiller

## Choose connection settings and an approved target

Set `CONFIG_PATH` to your verified NRC400 profile. This keeps USB adapter/unit identity bindings and verified serial framing/timeouts together. The current `[control]` section accepts only `expected_serial_number`; remove old `allow_writes`, `min_c` and `max_c` entries. The API has no write-enable switch, target range or state-based control lock. Choose targets for the actual coolant and experiment. There is no guessed COM port. `SIMULATE = True` ignores the hardware profile and opens no physical port; each run creates a fresh simulated device with no shared state. Review all operation settings before continuing.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config.toml"
SIMULATE = False
TARGET_C = 20.0  # Illustrative only. Choose a modest change approved for your setup.

## Connect, confirm, change one target, verify and release

Run this entire operation cell together: connection and cleanup share one context/finally block. Ctrl+C/Interrupt attempts cleanup while allowing Python to finish. Do not split the cell, kill the kernel during I/O, or rerun setup over an unfinished connection. Errors remain visible. Closing the port retains the equipment's target, loop and pump state.


In [ ]:
if isinstance(TARGET_C, bool) or not isfinite(TARGET_C):
    raise ValueError("TARGET_C must be a finite number in Celsius")

chiller = create_chiller(None if SIMULATE else CONFIG_PATH, simulate=SIMULATE)

with chiller:
    print(f"Backend: {chiller.transport.backend}; identity: {chiller.identity}")
    before = chiller.read_sample()
    print(f"Current target: {before.setpoint_c:.3f} C; supply: {before.temperature_c:.2f} C")
    print(f"Loop: {before.control_loop}; pump: {before.pump}; level: {before.level}")
    print("Check the filled/primed circuit, panel, coolant and approved temperature limits.")
    token = f"SET {TARGET_C:g}"
    if (
        input(f"Type {token!r} to send one target request; anything else cancels: ").strip()
        == token
    ):
        # Send once; this method already checks readback and device-unit rounding.
        confirmed_c = chiller.set_setpoint(TARGET_C)
        reported_c = chiller.read_setpoint()
        if abs(reported_c - confirmed_c) > 0.001:
            raise ProtocolError(
                "Target drifted after verification; inspect the panel. Do not retry."
            )
        print(f"Requested: {TARGET_C:g} C; confirmed rounded target: {reported_c:.3f} C")
        print("Compare with the panel. Readback does not establish thermal settling.")
    else:
        print("Cancelled; no target request sent.")

print("Connection closed; no automatic target restoration or equipment stop was sent.")

## Check results and finish

Compare the confirmed rounded target with the panel and observe supply temperature separately. Record any restoration as a separate intentional request. If the operation fails or is interrupted, inspect the equipment before reconnecting; the write may already have applied. Never replay an uncertain request.

**Software connection closure does not stop the chiller or pump.** Follow the physical shutdown procedure separately and confirm panel state. A forced kernel termination cannot guarantee cleanup. See [the hardware test record checklist](README.md#6-record-a-hardware-test-result) for evidence to retain.
